# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/surajprai2111-ui/ML-assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("hf_token")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("DuckDB connection created.")

DuckDB connection created.


In [4]:
rel = "hf://datasets/FlyRank/internship-warehouse"

print(rel)

hf://datasets/FlyRank/internship-warehouse


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item for one client on one report date in the
daily performance table.

For development, I will use the March 2026 slice rather than the final June
2026 month. The March slice contains 9,841,378 rows and covers
2026-03-01 to 2026-03-31.

The decision I am supporting is whether a content item should be flagged for
closer review because its observed search performance is declining.

The declining label is treated as a proxy outcome, not proof of future decline.
Features should represent information available before the outcome window, and
the label or fields derived from the outcome period will not be used as model
features.

The final June 2026 month will be treated as a sealed period for later
validation.

In [5]:
march_check = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
""").df()

display(march_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

| Bucket | Fields | Reason |
|---|---|---|
| Feature | `search_volume`, `competition`, `cpc`, `word_count` | Content-level fields from `dim_content` describing keyword demand, competition, cost-per-click, and content length. |
| Label / proxy | `is_declining_label` (derived) | A derived proxy for declining content based on the observed trend. It is the outcome being studied, not a model feature. |
| Context | `client_hash_id`, `content_hash_id`, `report_date` | Used for identification, grouping, joining, and time-based splitting, but not as model features. |
| Excluded | `trend_direction`, `trend_pct` | The declining label is derived from these fields, so using them as features would directly leak the target. |
| Excluded | Future-window performance fields | These are only known after the prediction point and would leak information from the outcome period. |
| Excluded | Raw client names, domains, URLs, queries, or other identifying information | Not needed for this analysis and should not be used as modeling inputs. |

The four proposed features are content-level fields from `dim_content`; they
would be joined to the daily performance data using `content_hash_id`.

I will check missingness before deciding how to handle blank feature values.
Missingness should not automatically be replaced with zero because it can
follow data availability or content-related patterns.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


The grain check returned no duplicate client-content-date combinations,
supporting the stated daily grain for the March 2026 slice.

In [11]:
date_check = con.execute(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS row_count
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
""").df()

display(date_check)

,min_date,max_date,row_count
0,2026-03-01,2026-03-31,9841378


The March development slice contains 9,841,378 daily rows and covers
2026-03-01 through 2026-03-31.

In [19]:
missingness_check = con.execute(f"""
WITH march_content AS (
    SELECT DISTINCT
        content_hash_id
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE report_date >= '2026-03-01'
      AND report_date < '2026-04-01'
)

SELECT
    COUNT(*) AS march_content_items,

    SUM(
        CASE WHEN d.search_volume IS NULL THEN 1 ELSE 0 END
    ) AS search_volume_missing,

    SUM(
        CASE WHEN d.competition IS NULL THEN 1 ELSE 0 END
    ) AS competition_missing,

    SUM(
        CASE WHEN d.cpc IS NULL THEN 1 ELSE 0 END
    ) AS cpc_missing,

    SUM(
        CASE WHEN d.word_count IS NULL THEN 1 ELSE 0 END
    ) AS word_count_missing

FROM march_content m

LEFT JOIN read_parquet(
    '{rel}/dim_content.parquet'
) d

ON m.content_hash_id = d.content_hash_id
""").df()

display(missingness_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_content_items,search_volume_missing,competition_missing,cpc_missing,word_count_missing
0,331437,59407.0,59407.0,59407.0,107429.0


### Missingness result

The March 2026 slice contains 331,437 unique content items. The keyword-related
fields `search_volume`, `competition`, and `cpc` each have 59,407 missing values
(17.92%). This shared missingness suggests that these fields are not missing
independently.

`word_count` has 107,429 missing values (32.42%), so its missingness pattern is
different from the keyword fields.

These results show that missing values should not automatically be replaced
with zero, because a blank value may represent unavailable or inapplicable
information rather than a true zero.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

The warehouse is an unbalanced panel, so different clients and content items
do not necessarily have the same amount of historical data.

Some earlier rows may contain search data without GA4 data because GA4
availability starts later for some clients. Therefore, a missing or unavailable
GA4 metric should not automatically be interpreted as zero engagement.

The query-level data uses a fixed 90-day window, which can overlap with recent
performance periods. Feature and label windows therefore need to be aligned
before query-level fields are used.

The current declining label is a proxy based on an observed trend. It does not
prove that a content item will decline in a future period or that refreshing
the content will improve performance.

The data supports measured and directional analysis, but it cannot establish
that a content change caused a performance improvement.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.